# Day 5 inference measurements on Kaggle

Attach the private `octlm-code` dataset and the completed `octlm-day4` notebook output. Run with one T4. This notebook measures KV-cached generation and int8 weight-only inference on the saved Day 4 checkpoint.


In [ ]:
import subprocess
import sys
from pathlib import Path

import torch

INPUT = Path("/kaggle/input")
CODE = next(path.parent.parent for path in INPUT.rglob("day5.py") if path.parent.name == "octlm")
TRAIN = next(path for path in INPUT.rglob("train.bin") if path.parent.name == "data")
DATA = TRAIN.parent
TOKENIZER = DATA.parent / "bpe.json"
CHECKPOINT = DATA.parent / "run.pt"
OUT = Path("/kaggle/working/day5")
OUT.mkdir(parents=True, exist_ok=True)
if not TOKENIZER.is_file() or not CHECKPOINT.is_file():
    raise FileNotFoundError("Attach the completed octlm-day4 notebook output")
if not torch.cuda.is_available() or torch.cuda.get_device_capability(0) != (7, 5):
    raise RuntimeError("Select a Kaggle T4 GPU")
print("PyTorch:", torch.__version__, "GPU:", torch.cuda.get_device_name(0), flush=True)


def run(stage, output):
    subprocess.run(
        [
            sys.executable,
            "-m",
            "octlm.day5",
            stage,
            "--config",
            str(CODE / "configs/day4.toml"),
            "--data",
            str(DATA),
            "--tokenizer",
            str(TOKENIZER),
            "--checkpoint",
            str(CHECKPOINT),
            "--out",
            str(OUT / output),
            "--device",
            "cuda:0",
        ],
        cwd=CODE,
        check=True,
    )
    print((OUT / output).read_text(), flush=True)

In [ ]:
run("cache", "day5-cache.jsonl")

In [ ]:
run("quant", "day5-quant.jsonl")